<a href="https://colab.research.google.com/github/Segn11/datasciencebootcamp_project/blob/sypto_5/Ucrpto20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [11]:
df = pd.read_csv("/content/Train (8).csv")
test = pd.read_csv("/content/Test (7).csv")

In [12]:
df.head()

,id,asset_id,open,high,low,volume,market_cap,url_shares,unique_url_shares,reddit_posts,...,percent_change_24h_rank,volume_24h_rank,social_volume_24h_rank,social_score_24h_rank,medium,youtube,social_volume,percent_change_24h,market_cap_global,close
0,ID_322qz6,1,9422.849081,9428.490628,9422.849081,7.131986e+08,1.737635e+11,1689.0,817.0,55.0,...,606.0,2.0,1.0,1.0,2.0,5.0,4422,1.434516,2.818066e+11,9428.279323
1,ID_3239o9,1,7985.359278,7992.059917,7967.567267,4.004755e+08,1.426942e+11,920.0,544.0,20.0,...,NaN,NaN,NaN,NaN,NaN,NaN,2159,-2.459507,2.126897e+11,7967.567267
2,ID_323J9k,1,49202.033778,49394.593518,49068.057046,3.017729e+09,9.166977e+11,1446.0,975.0,72.0,...,692.0,3.0,1.0,1.0,NaN,NaN,10602,4.942448,1.530712e+12,49120.738484
3,ID_323y5P,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,17.0,...,NaN,NaN,NaN,NaN,NaN,NaN,285,NaN,NaN,NaN
4,ID_324kJH,1,10535.737119,10535.737119,10384.798216,1.150053e+09,1.921183e+11,1012.0,638.0,24.0,...,749.0,2.0,1.0,1.0,NaN,2.0,3996,2.609576,3.386925e+11,10384.798216


In [13]:
df= df.fillna(0)
test = test.fillna(0)

In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12632 entries, 0 to 12631
Data columns (total 49 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       12632 non-null  object 
 1   asset_id                 12632 non-null  int64  
 2   open                     12632 non-null  float64
 3   high                     12632 non-null  float64
 4   low                      12632 non-null  float64
 5   volume                   12632 non-null  float64
 6   market_cap               12632 non-null  float64
 7   url_shares               12632 non-null  float64
 8   unique_url_shares        12632 non-null  float64
 9   reddit_posts             12632 non-null  float64
 10  reddit_posts_score       12632 non-null  float64
 11  reddit_comments          12632 non-null  float64
 12  reddit_comments_score    12632 non-null  float64
 13  tweets                   12632 non-null  float64
 14  tweet_spam            

In [16]:
features = [
    'open', 'high', 'low', 'market_cap', 'market_cap_global'
]
target = ['close']


In [18]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

estimator = RandomForestRegressor()
selector = RFE(estimator, n_features_to_select=6, step=1)
selector = selector.fit(df[features], df[target])
features_selected = selector.support_

/usr/local/lib/python3.12/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example usi

In [19]:
print("Selected Features:", features_selected)

Selected Features: [ True  True  True  True  True False False False  True]


In [30]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor

# -----------------------------
# LOAD DATA
# -----------------------------
train = pd.read_csv("/content/Train (8).csv")
test  = pd.read_csv("/content/Test (7).csv")

train.fillna(0, inplace=True)
test.fillna(0, inplace=True)

# -----------------------------
# FEATURE ENGINEERING
# -----------------------------
def add_features(df, is_train=True):
    # Safe ratios (avoid division by zero)
    df['high_low_ratio'] = df['high'] / df['low'].replace(0, np.nan)
    df['close_open_ratio'] = df['open'] / df['low'].replace(0, np.nan)
    df['high_open_ratio'] = df['high'] / df['open'].replace(0, np.nan)

    # Log transforms (safe)
    df['volume_log'] = np.log1p(df['volume'].clip(lower=0))
    df['market_cap_log'] = np.log1p(df['market_cap'].clip(lower=0))
    df['market_cap_global_log'] = np.log1p(df['market_cap_global'].clip(lower=0))

    # Rolling features (volume only, safe for test)
    df['vol_change'] = df['volume'].pct_change().fillna(0)
    df['vol_ma_3'] = df['volume'].rolling(3).mean().fillna(method='bfill')

    # Target-dependent features only for train
    if is_train:
        df['return_1'] = df['close'].diff().fillna(0)
        df['return_3'] = df['close'].diff(3).fillna(0)
        df['ma_3'] = df['close'].rolling(3).mean().fillna(method='bfill')
        df['ma_7'] = df['close'].rolling(7).mean().fillna(method='bfill')

    # Replace any remaining inf/-inf with 0
    df.replace([np.inf, -np.inf], 0, inplace=True)

    return df

train = add_features(train, is_train=True)
test  = add_features(test, is_train=False)

# -----------------------------
# FEATURES & TARGET
# -----------------------------
features = [
    'open','high','low',
    'market_cap','market_cap_global','volume',
    'high_low_ratio','close_open_ratio','high_open_ratio',
    'volume_log','market_cap_log','market_cap_global_log',
    'vol_change','vol_ma_3'
]
target = 'close'

X = train[features]
y = train[target]

# Replace any remaining inf/-inf in train/test
X.replace([np.inf, -np.inf], 0, inplace=True)
test[features].replace([np.inf, -np.inf], 0, inplace=True)

# -----------------------------
# K-FOLD CROSS-VALIDATION
# -----------------------------
kf = KFold(n_splits=5, shuffle=True, random_state=42)
rmse_scores, mae_scores = [], []

fold = 1
for train_idx, val_idx in kf.split(X):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = RandomForestRegressor(
        n_estimators=800,
        max_depth=18,
        min_samples_split=3,
        min_samples_leaf=2,
        n_jobs=-1,
        random_state=42
    )

    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)

    mae = mean_absolute_error(y_val, y_pred)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    print(f"Fold {fold} → MAE: {mae:.4f}, RMSE: {rmse:.4f}")

    mae_scores.append(mae)
    rmse_scores.append(rmse)
    fold += 1

print("\n==== FINAL K-FOLD RESULTS ====")
print(f"Average RMSE: {np.mean(rmse_scores):.4f}")

# -----------------------------
# TRAIN FINAL MODEL
# -----------------------------
final_model = RandomForestRegressor(
    n_estimators=800,
    max_depth=18,
    min_samples_split=3,
    min_samples_leaf=2,
    n_jobs=-1,
    random_state=42
)
final_model.fit(X, y)

# -----------------------------
# PREDICT TEST SET
# -----------------------------
test["pred_close"] = final_model.predict(test[features])

# -----------------------------
# CREATE SUBMISSION FILE
# -----------------------------
if "id" in test.columns:
    submission = pd.DataFrame({"id": test["id"], "close": test["pred_close"]})
else:
    submission = pd.DataFrame({"index": test.index, "close": test["pred_close"]})

submission_path = "/content/submission_rf_robust.csv"
submission.to_csv(submission_path, index=False)
print(f"Submission saved to: {submission_path}")


/tmp/ipython-input-3362258285.py:32: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df['vol_ma_3'] = df['volume'].rolling(3).mean().fillna(method='bfill')
/tmp/ipython-input-3362258285.py:38: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df['ma_3'] = df['close'].rolling(3).mean().fillna(method='bfill')
/tmp/ipython-input-3362258285.py:39: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df['ma_7'] = df['close'].rolling(7).mean().fillna(method='bfill')
/tmp/ipython-input-3362258285.py:32: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df['vol_ma_3'] = df['volume'].rolling(3).mean().fillna(method='bfill')
/tmp/ipython-input-3362258285.py:65: S

Fold 1 → MAE: 17.8052, RMSE: 58.9881
Fold 2 → MAE: 19.7676, RMSE: 59.9662
Fold 3 → MAE: 19.2962, RMSE: 66.2221
Fold 4 → MAE: 19.2130, RMSE: 54.8099
Fold 5 → MAE: 17.2928, RMSE: 47.5926

==== FINAL K-FOLD RESULTS ====
Average MAE:  18.6750
Average RMSE: 57.5158
Submission saved to: /content/submission_rf_robust.csv


In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor

# -----------------------------
# LOAD DATA
# -----------------------------
train = pd.read_csv("/content/Train (8).csv")
test  = pd.read_csv("/content/Test (7).csv")

train.fillna(0, inplace=True)
test.fillna(0, inplace=True)

# -----------------------------
# FEATURE ENGINEERING
# -----------------------------
def add_features(df, is_train=True):
    # Safe ratios (avoid division by zero)
    df['high_low_ratio'] = df['high'] / df['low'].replace(0, np.nan)
    df['close_open_ratio'] = df['open'] / df['low'].replace(0, np.nan)
    df['high_open_ratio'] = df['high'] / df['open'].replace(0, np.nan)

    # Log transforms (safe)
    df['volume_log'] = np.log1p(df['volume'].clip(lower=0))
    df['market_cap_log'] = np.log1p(df['market_cap'].clip(lower=0))
    df['market_cap_global_log'] = np.log1p(df['market_cap_global'].clip(lower=0))

    # Rolling features (volume only, safe for test)
    df['vol_change'] = df['volume'].pct_change().fillna(0)
    df['vol_ma_3'] = df['volume'].rolling(3).mean().fillna(method='bfill')

    # Target-dependent features only for train
    if is_train:
        df['return_1'] = df['close'].diff().fillna(0)
        df['return_3'] = df['close'].diff(3).fillna(0)
        df['ma_3'] = df['close'].rolling(3).mean().fillna(method='bfill')
        df['ma_7'] = df['close'].rolling(7).mean().fillna(method='bfill')

    # Replace any remaining inf/-inf with 0
    df.replace([np.inf, -np.inf], 0, inplace=True)

    return df

train = add_features(train, is_train=True)
test  = add_features(test, is_train=False)

# -----------------------------
# FEATURES & TARGET
# -----------------------------
features = [
    'open','high','low',
    'market_cap','market_cap_global','volume',
    'high_low_ratio','close_open_ratio','high_open_ratio',
    'volume_log','market_cap_log','market_cap_global_log',
    'vol_change','vol_ma_3'
]
target = 'close'

X = train[features]
y = train[target]

/tmp/ipython-input-1301070358.py:32: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df['vol_ma_3'] = df['volume'].rolling(3).mean().fillna(method='bfill')
/tmp/ipython-input-1301070358.py:38: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df['ma_3'] = df['close'].rolling(3).mean().fillna(method='bfill')
/tmp/ipython-input-1301070358.py:39: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df['ma_7'] = df['close'].rolling(7).mean().fillna(method='bfill')
/tmp/ipython-input-1301070358.py:32: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df['vol_ma_3'] = df['volume'].rolling(3).mean().fillna(method='bfill')


In [8]:
from sklearn.ensemble import RandomForestRegressor
import pandas as pd
import numpy as np

# Fit a RandomForest to get feature importances
rf = RandomForestRegressor()
rf.fit(train[features], train[target])

# Get feature importances
importances = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)

# Select only important features (you can choose threshold)
important_features = importances[importances > 0.01].index.tolist()
print("Important Features:", important_features)


Important Features: ['low', 'high', 'open', 'market_cap_log', 'market_cap', 'market_cap_global', 'market_cap_global_log']


In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv("/content/Train (8).csv")
test = pd.read_csv("/content/Test (7).csv")

In [ ]:
features = ['low', 'high', 'open', 'market_cap_log', 'market_cap',
            'market_cap_global', 'market_cap_global_log']

 target = ['close']

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error

# -----------------------------
# LOAD DATA
# -----------------------------
df = pd.read_csv("/content/Train (8).csv")
test = pd.read_csv("/content/Test (7).csv")

# Fill missing values
df.fillna(0, inplace=True)
test.fillna(0, inplace=True)

# -----------------------------
# FEATURE ENGINEERING
# -----------------------------
def add_features(df, is_train=True):

    # Log transforms (safe)

    df['market_cap_log'] = np.log1p(df['market_cap'].clip(lower=0))
    df['market_cap_global_log'] = np.log1p(df['market_cap_global'].clip(lower=0))
    df.replace([np.inf, -np.inf], 0, inplace=True)

    return df

df = add_features(df, is_train=True)
test  = add_features(test, is_train=False)

df.fillna(0, inplace=True)
test.fillna(0, inplace=True)

# -----------------------------

# FEATURES & TARGET
# -----------------------------
features = ['low', 'high', 'open', 'market_cap_log', 'market_cap',
            'market_cap_global', 'market_cap_global_log']
target = 'close'

X = df[features]
y = df[target]

# -----------------------------
# K-Fold Cross Validation
# -----------------------------
kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold = 1
rmse_list = []
mae_list = []

for train_index, val_index in kf.split(X):
    X_train, X_val = X.iloc[train_index], X.iloc[val_index]
    y_train, y_val = y.iloc[train_index], y.iloc[val_index]

    model = RandomForestRegressor(n_estimators=200, random_state=42)
    model.fit(X_train, y_train)

    y_val_pred = model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
    mae = mean_absolute_error(y_val, y_val_pred)

    print(f"Fold {fold} - RMSE: {rmse:.3f}")
    rmse_list.append(rmse)
    #mae_list.append(mae)

    fold += 1

print(f"\nAverage RMSE: {np.mean(rmse_list):.3f}")
#print(f"Average MAE: {np.mean(mae_list):.3f}")

# -----------------------------
# TRAIN ON FULL DATA & PREDICT TEST
# -----------------------------
final_model = RandomForestRegressor(n_estimators=200, random_state=42)
final_model.fit(X, y)
test_predictions = final_model.predict(test[features])


Fold 1 - RMSE: 59.116
Fold 2 - RMSE: 59.464
Fold 3 - RMSE: 68.386
Fold 4 - RMSE: 53.345
Fold 5 - RMSE: 47.974

Average RMSE: 57.657


In [14]:
submission = pd.DataFrame({
    "id": test["id"],
    "close": test_predictions  # use the array, not a column
})
submission.to_csv("submission.csv", index=False)
print("Saved submission.")


Saved submission.


In [2]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

estimator = RandomForestRegressor()
selector = RFE(estimator, n_features_to_select=6, step=1)
selector = selector.fit(df[features], df[target])
features_selected = selector.support_

In [44]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestRegressor

# List of features
features = [
    'open', 'high', 'low', 'market_cap', 'market_cap_global', 'dominance',
    'market_cap_change', 'market_cap_global_change', 'cap_to_global_ratio',
    'cap_change_vs_global'
]

# Create features safely
for df_ in [df, test]:
    # Dominance
    df_['dominance'] = df_['market_cap'] / df_['market_cap_global']

    # Market cap changes
    df_['market_cap_change'] = df_['market_cap'].pct_change().fillna(0)
    df_['market_cap_global_change'] = df_['market_cap_global'].pct_change().fillna(0)

    # Ratios
    df_['cap_to_global_ratio'] = df_['market_cap'] / df_['market_cap_global']

    # Safe division for relative change
    df_['cap_change_vs_global'] = df_['market_cap_change'] / df_['market_cap_global_change'].replace(0, np.nan)
    df_['cap_change_vs_global'] = df_['cap_change_vs_global'].replace([np.inf, -np.inf], 0).fillna(0)

# Clip extremely large values to prevent overflow
clip_limit = 1e12  # adjust if needed
for col in features:
    df[col] = df[col].clip(-clip_limit, clip_limit)
    test[col] = test[col].clip(-clip_limit, clip_limit)

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[features])

# Flatten target
y = df['close'].values

# RFE feature selection
estimator = RandomForestRegressor()
selector = RFE(estimator, n_features_to_select=5, step=1)
selector = selector.fit(X_scaled, y)

features_selected = [f for f, s in zip(features, selector.support_) if s]
print("Selected Features:", features_selected)


Selected Features: ['open', 'high', 'low', 'market_cap', 'market_cap_global']


In [ ]:
df["date"] = pd.date_range(start="2020-01-01", periods=len(df), freq="D")
test["date"] = pd.date_range(start="2020-01-01", periods=len(test), freq="D")


In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor

# -------------------------------------
# LOAD DATA
# -------------------------------------
train = pd.read_csv("/content/Train (8).csv")
test  = pd.read_csv("/content/Test (7).csv")

train.fillna(0, inplace=True)
test.fillna(0, inplace=True)

# -------------------------------------
# FEATURE ENGINEERING
# -------------------------------------
def add_features(df, is_train=True):
    df["high_low_ratio"] = df["high"] / df["low"].replace(0, np.nan)
    df["close_open_ratio"] = df["open"] / df["low"].replace(0, np.nan)
    df["high_open_ratio"] = df["high"] / df["open"].replace(0, np.nan)
    df["low_open_ratio"] = df["low"] / df["open"].replace(0, np.nan)

    df["wick_upper"] = df["high"] - df[["open", "low"]].max(axis=1)
    df["wick_lower"] = df[["open", "low"]].min(axis=1) - df["low"]
    df["body"] = df["low"] - df["open"]
    df["body_ratio"] = df["body"] / (df["high"] - df["low"] + 1e-9)

    df["volume_log"] = np.log1p(df["volume"].clip(lower=0))
    df["market_cap_log"] = np.log1p(df["market_cap"].clip(lower=0))
    df["market_cap_global_log"] = np.log1p(df["market_cap_global"].clip(lower=0))

    df["open_x_volume"] = df["open"] * df["volume"]
    df["high_x_volume"] = df["high"] * df["volume"]
    df["marketcap_x_volume"] = df["market_cap"] * df["volume"]

    df["volume_relative"] = df["volume"] / (df["market_cap"] + 1e-9)
    df["price_relative_global"] = df["market_cap"] / (df["market_cap_global"] + 1e-9)

    df["volatility_like"] = abs(df["high"] - df["low"])
    df["spread"] = abs(df["open"] - df["low"])

    df.replace([np.inf, -np.inf], 0, inplace=True)
    return df

train = add_features(train, True)
test  = add_features(test, False)

# -------------------------------------
# FEATURES AND TARGET
# -------------------------------------
target = "close"

features = [
    'open','high','low','volume',
    'market_cap','market_cap_global',

    'high_low_ratio','close_open_ratio','high_open_ratio','low_open_ratio',
    'wick_upper','wick_lower','body','body_ratio',

    'volume_log','market_cap_log','market_cap_global_log',

    'open_x_volume','high_x_volume','marketcap_x_volume',
    'volume_relative','price_relative_global',

    'volatility_like','spread'
]

X = train[features]
y = train[target]
test_X = test[features]

# -------------------------------------
# K-FOLD TRAINING (ONLY RANDOM FOREST)
# -------------------------------------
kf = KFold(n_splits=5, shuffle=True, random_state=42)
rf_preds = np.zeros(len(test))

fold = 1
for tr, val in kf.split(X):
    X_train, X_val = X.iloc[tr], X.iloc[val]
    y_train, y_val = y.iloc[tr], y.iloc[val]

    rf = RandomForestRegressor(
        n_estimators=800,
        max_depth=18,
        min_samples_split=3,
        min_samples_leaf=2,
        n_jobs=-1,
        random_state=42
    )

    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_val)

    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    mae = mean_absolute_error(y_val, y_pred)
    print(f"[Fold {fold}] RF → RMSE: {rmse:.4f}, MAE: {mae:.4f}")

    rf_preds += rf.predict(test_X) / kf.n_splits
    fold += 1

# -------------------------------------
# FINAL PREDICTION
# -------------------------------------
test["pred_close"] = rf_preds

# -------------------------------------
# CREATE SUBMISSION
# -------------------------------------
if "id" in test.columns:
    submission = pd.DataFrame({"id": test["id"], "close": test["pred_close"]})
else:
    submission = pd.DataFrame({"index": test.index, "close": test["pred_close"]})

submission_path = "/content/submission_rf_only.csv"
submission.to_csv(submission_path, index=False)
print(f"\nSubmission saved to: {submission_path}")


[Fold 1] RF → RMSE: 58.2360, MAE: 17.5473
[Fold 2] RF → RMSE: 59.9117, MAE: 19.6043
[Fold 3] RF → RMSE: 67.5549, MAE: 19.4006
[Fold 4] RF → RMSE: 54.2302, MAE: 19.0639
[Fold 5] RF → RMSE: 47.2223, MAE: 17.0844

Submission saved to: /content/submission_rf_only.csv
